In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Cargamos el dataset maestro
df = pd.read_csv('./data/titanic_model_ready.csv')

# Definimos X (características) y y (lo que queremos predecir)
X = df.drop('Survived', axis=1)
y = df['Survived']

# Dividimos: 80% para aprender, 20% para examen final
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Entrenamiento: {X_train.shape[0]} pasajeros, Prueba: {X_test.shape[0]} pasajeros")

Entrenamiento: 712 pasajeros, Prueba: 179 pasajeros


In [2]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

# Diccionario de modelos con sus "perillas" (hiperparámetros)
modelos_config = {
    'Logistic Regression': {
        'model': LogisticRegression(max_iter=1000),
        'params': {'C': [0.1, 1, 10]}
    },
    'Random Forest': {
        'model': RandomForestClassifier(),
        'params': {'n_estimators': [10, 50, 100], 'max_depth': [None, 5, 10]}
    },
    'KNN': {
        'model': KNeighborsClassifier(),
        'params': {'n_neighbors': [3, 5, 11]}
    }
}

In [3]:
import pickle

resultados = []

for nombre, config in modelos_config.items():
    print(f"Entrenando {nombre}...")
    # GridSearchCV prueba todas las combinaciones y hace Cross-Validation
    gs = GridSearchCV(config['model'], config['params'], cv=5, return_train_score=False)
    gs.fit(X_train, y_train)
    
    resultados.append({
        'modelo': nombre,
        'mejor_score': gs.best_score_,
        'mejor_params': gs.best_params_
    })

# Convertimos a DataFrame para ver quién ganó
df_resultados = pd.DataFrame(resultados).sort_values(by='mejor_score', ascending=False)
print("\n--- Resultados de la Competencia ---")
print(df_resultados)

# Guardamos el mejor modelo de todos (el ganador)
mejor_modelo_obj = gs.best_estimator_ # Aquí guardamos el ganador de la última iteración o el que elijas
with open('modelo_titanic.pkl', 'wb') as f:
    pickle.dump(mejor_modelo_obj, f)

Entrenando Logistic Regression...
Entrenando Random Forest...
Entrenando KNN...

--- Resultados de la Competencia ---
                modelo  mejor_score                           mejor_params
1        Random Forest     0.823008  {'max_depth': 10, 'n_estimators': 50}
2                  KNN     0.800542                     {'n_neighbors': 3}
0  Logistic Regression     0.796287                             {'C': 0.1}
